# Entraînement sur Kaggle (GPU)

Ce notebook reproduit l'entraînement principal sur un GPU Kaggle :
1. récupère le code depuis le dépôt GitHub (privé, via un jeton stocké dans les *Secrets* Kaggle) ;
2. télécharge les 20 fichiers de simulations depuis le Google Drive de coEnergy (ou utilise un *Dataset* Kaggle s'il est attaché) ;
3. lance `scripts/train.py` avec une configuration YAML ;
4. archive les résultats (`runs.zip`) pour les rapatrier dans `artifacts/runs/` du dépôt.

**Réglages requis** (panneau de droite) : *Accelerator* = GPU, *Internet* = activé, secret `GITHUB_TOKEN` attaché.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
# ---- 1. Code -------------------------------------------------------------
from kaggle_secrets import UserSecretsClient

GITHUB_REPO = "Fareshd049/coenergy-assessment"
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!rm -rf /kaggle/working/repo
!git clone -q https://{token}@github.com/{GITHUB_REPO}.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
# torch est déjà installé sur Kaggle : on installe le package seul, sans réinstaller ses dépendances
!pip install -q --no-deps -e .
import sys

sys.path.insert(0, "/kaggle/working/repo/src")  # le noyau déjà lancé ne voit pas les paquets installés après son démarrage
import torch

print("hlcnet OK | torch", torch.__version__, "| CUDA", torch.cuda.is_available())

In [ ]:
# ---- 2. Données ----------------------------------------------------------
from pathlib import Path


def find_batches(root: Path) -> Path | None:
    hits = sorted(root.rglob("batch_1.pt")) if root.exists() else []
    return hits[0].parent if hits else None

DATA = find_batches(Path("/kaggle/input"))  # Dataset Kaggle attaché, si présent
if DATA is None:
    DRIVE_FOLDER = "https://drive.google.com/drive/folders/1shkMLj_NnCwKhh27KAlh9rj5Y_h68MEW"  # quentin_house
    !pip install -q gdown
    !gdown --folder {DRIVE_FOLDER} -O /kaggle/working/data --remaining-ok
    DATA = find_batches(Path("/kaggle/working/data"))

files = sorted(DATA.glob("*.pt"))
print(DATA, "|", len(files), "fichiers |", round(sum(f.stat().st_size for f in files) / 1e9, 2), "Go")
assert len(files) == 20, "il manque des fichiers : relancer la cellule"

In [ ]:
# ---- 3. Entraînement -----------------------------------------------------
# Choisir la liste d'expériences à lancer (exécutées l'une après l'autre).
ABLATIONS = [
    "configs/ablations/abl_head_sum.yaml",
    "configs/ablations/abl_head_free.yaml",
    "configs/ablations/abl_mlp.yaml",
    "configs/ablations/abl_no_deriv.yaml",
    "configs/ablations/abl_mse.yaml",
    "configs/ablations/abl_noise.yaml",
]
ARCHITECTURES = [
    "configs/ablations/abl_tcn_gru.yaml",
    "configs/ablations/abl_tcn.yaml",
    "configs/ablations/abl_film_gru.yaml",
]
DIAGNOSTIC = ["configs/ablations/abl_no_rounding.yaml"]
NOISE = [
    "configs/ablations/abl_noise005.yaml",
    "configs/ablations/abl_noise005_noderiv.yaml",
    "configs/ablations/abl_noise_rand010.yaml",
]
FINAL = ["configs/final_700k.yaml", "configs/final_700k_noise005.yaml"]

CONFIGS = ARCHITECTURES  # <-- ABLATIONS, ARCHITECTURES, NOISE, DIAGNOSTIC ou FINAL selon la version lancée
RUN_ROCKET = True        # baseline ROCKET + ridge (sur les données de configs/full.yaml)

for cfg in CONFIGS:
    print("=" * 100, "\n", cfg, "\n", "=" * 100)
    !python scripts/train.py --config {cfg} --data-folder {DATA} --out /kaggle/working/runs

if RUN_ROCKET:
    print("=" * 100, "\n ROCKET + ridge\n", "=" * 100)
    !python scripts/rocket_baseline.py --config configs/full.yaml --data-folder {DATA} --out /kaggle/working/runs --n-kernels 2000

In [ ]:
# ---- 4. Archive des résultats --------------------------------------------
!cd /kaggle/working && zip -qr runs.zip runs && ls -lh runs.zip